# Structured Output 

Models can be requested to provide their response in a format matching a given schema. This is userful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

# Pydantic

Pydantic models provides the richest feature set with field validation, descriptions, and nested structures.

In [2]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10f646db0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10eaeddc0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The Title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="the director of the movie")
    Leadactor:str=Field(description="the Lead actor of the movie")
    rating:float=Field(description="The movis rating out of 10")

In [11]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x11343c3e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11352fad0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'pro

In [18]:
model_with_structure.invoke("Provide details about titanic movie")

Movie(title='Titanic', year=1997, director='James Cameron', Leadactor='Leonardo DiCaprio', rating=7.8)

### Message Output alongside parsed strcuture 

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(...,description="The Title of the movie")
    year:int=Field(...,description="This year movie was released")
    director:str=Field(...,description="the director of the movie")
    Leadactor:str=Field(...,description="the Lead actor of the movie")
    rating:float=Field(...,description="The movis rating out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about titanic movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about Titanic. Likely use the function. The function expects Leadactor, director, rating, title, year. Provide details. Probably call the function.', 'tool_calls': [{'id': 'fc_cf7c8df6-5c1a-4509-a837-1d5b9a40cfa4', 'function': {'arguments': '{"Leadactor":"Leonardo DiCaprio","director":"James Cameron","rating":8.8,"title":"Titanic","year":1997}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 169, 'total_tokens': 254, 'completion_time': 0.09249488, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.008408825, 'prompt_tokens_details': None, 'queue_time': 0.194201803, 'total_time': 0.100903705}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fee7f-4657-7411

### Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None,description= "budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)





In [12]:
response = model_with_structure.invoke("Provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160000000.0)

# TypedDict

TypedDict provides a simple alternative usign python built in typing, ideal when you dont need runtime validation

In [17]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was realsed"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("please provide the details of the movie avatar")
response


{'director': 'James Cameron', 'rating': 7.9, 'title': 'Avatar', 'year': 2009}

# DataClasses

A Data class is class typcially contailning mainly data, althrough there are'nt really any restrictions. You create it using the @dataclass decorator

In [18]:
import os 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email adress of the person")
    phone: str = Field(description= "The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo

)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result['structured_response'])


name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [22]:
result  

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='b9c772a5-25ea-4006-8fd7-4040926894c1'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 868, 'prompt_tokens': 204, 'total_tokens': 1072, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EBWhUGy9SwArMU3qyqRxmj9dql2DR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019feeb1-2a34-7f70-8697-86e310797895-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'outpu

In [24]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [27]:
##Typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):

    name: str
    email : str
    phone : str

agnet = create_agent(
    model = "gpt-5",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from : John Doe, John@explain.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='John@explain.com', phone='(555) 123-4567')

In [28]:
## DataClass


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo :
    """Contact infomration for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model ="gpt-5",
    response_format = ContactInfo
)

result["structured_response"]

ContactInfo(name='John Doe', email='John@explain.com', phone='(555) 123-4567')